# Stage B — Model Comparison on Public Benchmarks

Compares **our models** against **published research methods** on a *within-dataset* protocol: every model
is trained and tested on the **same** benchmark split (paper-disjoint 80/20), so the comparison is fair —
unlike the internal-gold eval, which is cross-domain.

## Which datasets are valid for this (per-sentence classification)?

**Used** — each has per-sentence labels, a gap-relevant class, negatives, and a test protocol:
| dataset | task | published SOTA |
|---|---|---|
| **FWS** (Zhang 2022) | binary future-work | 0.907 macro-F1 *(leakage-inflated)* → **0.825 honest** |
| **SAL/RCT** (Lan 2024) | binary limitation | PubMedBERT **~0.81** |
| **Sci-Sentence** (Bolaños 2025) | 7-class incl. gap/limitation/extension | fine-tuned **>0.96** (their split + augmentation) |

**Excluded, with reasons:**
- **LimGen, BAGELS** — *generation / coverage* tasks, not per-sentence classification → invalid for our task.
- **CSAbstruct, PubMed-RCT, csabstracts** — per-sentence but **no gap/limitation class** (only abstract roles) → usable only as *negatives*, not a gap benchmark.
- **ALTA-2019, LREC-2012** — the only other sets with an explicit gap/limitation sentence class, but both download links are **dead** (need author contact).

**Do we need more?** No new dataset is required for a per-sentence gap/limitation/future-work comparison —
these three cover both target classes with published baselines. More limitation-labeled, *in-domain (arXiv/CS)*
data would help our real pipeline, but no suitable public one is currently available (hence SAL's biomedical stand-in).

## Setup

In [ ]:
import re, json, warnings
import numpy as np, pandas as pd
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT=Path.cwd()
while not (ROOT/"data"/"external").exists() and ROOT!=ROOT.parent: ROOT=ROOT.parent
EXT=ROOT/"data"/"external"; print("data:",EXT)

_FUT=re.compile(r"\b(future work|future direction|future research|future stud(?:y|ies)|in future|for future|as future|future investigation|we\s+(?:plan|intend|aim|hope|would like|will)\s+to|next step|further (?:work|research|investigation|study)|would be (?:interesting|valuable|worthwhile)|it would be interesting)\b",re.I)
_LIM=re.compile(r"\b(limitation|shortcoming|drawback|weakness|caveat|we did not|we do not|we have not|is (?:restricted|limited|confined) to|only (?:considers?|applies?|works?|handles?)|unexplored|unaddressed|beyond the scope|out of scope|(?:cannot|fail[s]? to|do(?:es)? not (?:generalize|scale|extend))|lack[s]? (?:of|a )|(?:suffers? from|prone to|sensitive to))\b",re.I)

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.naive_bayes import BernoulliNB
from sklearn.pipeline import Pipeline
import torch
from sentence_transformers import SentenceTransformer
DEV="cuda" if torch.cuda.is_available() else "cpu"; print("device:",DEV)
RUN_FROZEN_FWS=False   # FWS has 18k sents -> heavy to embed on CPU; True on GPU
RUN_SCIBERT=False      # fine-tune SciBERT per benchmark; needs GPU (torch>=2.6)
_enc={}
def emb(name,ss):
    if name not in _enc: _enc[name]=SentenceTransformer(name,device=DEV)
    return np.asarray(_enc[name].encode(list(ss),normalize_embeddings=True,batch_size=512,show_progress_bar=False))
def split(df,seed=0):
    tri,tei=next(GroupShuffleSplit(1,test_size=0.2,random_state=seed).split(df,df.y,df.group)); return df.iloc[tri].reset_index(drop=True), df.iloc[tei].reset_index(drop=True)

In [ ]:
def scibert_ft(tr,te,labels,epochs=3):
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    import collections
    L2I={l:i for i,l in enumerate(labels)}; I2L={i:l for l,i in L2I.items()}
    tok=AutoTokenizer.from_pretrained("allenai/scibert_scivocab_uncased")
    m=AutoModelForSequenceClassification.from_pretrained("allenai/scibert_scivocab_uncased",num_labels=len(labels)).to(DEV)
    cnt=collections.Counter(tr.y); w=torch.tensor([len(tr)/(len(labels)*cnt[I2L[i]]) for i in range(len(labels))],dtype=torch.float,device=DEV)
    lossf=torch.nn.CrossEntropyLoss(weight=w); opt=torch.optim.AdamW(m.parameters(),lr=2e-5); sc=torch.cuda.amp.GradScaler()
    ts=tr.text.tolist(); ty=[L2I[x] for x in tr.y]
    for ep in range(epochs):
        m.train(); perm=np.random.permutation(len(ts))
        for i in range(0,len(perm),32):
            idx=perm[i:i+32]; b=tok([ts[j] for j in idx],truncation=True,max_length=96,padding=True,return_tensors="pt").to(DEV)
            yy=torch.tensor([ty[j] for j in idx],device=DEV); opt.zero_grad()
            with torch.cuda.amp.autocast(): loss=lossf(m(**b).logits,yy)
            sc.scale(loss).backward(); sc.step(opt); sc.update()
    m.eval(); out=[]
    with torch.no_grad():
        for i in range(0,len(te),128):
            b=tok(te.text.tolist()[i:i+128],truncation=True,max_length=96,padding=True,return_tensors="pt").to(DEV)
            with torch.cuda.amp.autocast(): out+=[I2L[int(x)] for x in m(**b).logits.argmax(-1).cpu()]
    return out

def compare(tr,te,labels,pos=None,rules_re=None,frozen=True):
    res={}
    if rules_re is not None and pos:
        res["cue-rules"]=[pos if rules_re.search(s) else "neg" for s in te.text]
    if frozen:
        for nm,hf in [("frozen bge-small","BAAI/bge-small-en-v1.5"),("frozen MiniLM","sentence-transformers/all-MiniLM-L6-v2")]:
            clf=LogisticRegression(C=2.0,max_iter=2000,class_weight="balanced").fit(emb(hf,tr.text),tr.y); res[nm]=list(clf.predict(emb(hf,te.text)))
    res["TF-IDF+SVM"]=Pipeline([("v",TfidfVectorizer(ngram_range=(1,2),min_df=2,max_features=50000)),("c",LinearSVC(class_weight="balanced"))]).fit(tr.text,tr.y).predict(te.text).tolist()
    res["BoW+chi2+BernoulliNB"]=Pipeline([("v",CountVectorizer(binary=True,ngram_range=(1,2),min_df=2)),("k",SelectKBest(chi2,k=min(3000,20000))),("c",BernoulliNB())]).fit(tr.text,tr.y).predict(te.text).tolist()
    if RUN_SCIBERT:
        try: res["SciBERT-FT"]=scibert_ft(tr,te,labels)
        except Exception as e: print("SciBERT-FT failed:",str(e)[:100])
    rows=[]
    for nm,p in res.items():
        r={"model":nm,"macro_F1":round(f1_score(te.y,p,labels=labels,average="macro",zero_division=0),3)}
        if pos: r["pos_F1"]=round(f1_score(te.y,p,labels=[pos],average="macro",zero_division=0),3)
        rows.append(r)
    return pd.DataFrame(rows).set_index("model")

## Benchmark 1 — FWS (binary future-work)
Zhang protocol: balanced 1:1, paper-disjoint. SOTA 0.907 (leaky) / 0.825 (honest).

In [ ]:
fws=pd.read_csv(EXT/"fws_corpus.csv")
pos=fws[fws.label==1]; neg=fws[fws.label==0].sample(len(pos),random_state=0)
d=pd.concat([pos,neg]); d=pd.DataFrame({"text":d.text.astype(str),"y":np.where(d.label==1,"future_work","neg"),"group":d.id.astype(str)})
tr,te=split(d); print("FWS balanced",len(d),"test",len(te))
compare(tr,te,["neg","future_work"],"future_work",_FUT,frozen=RUN_FROZEN_FWS)

**Reference (H100, full run):**
| model | macro_F1 | pos_F1 |
|---|---|---|
| cue-rules | 0.676 | 0.585 |
| frozen bge-small | 0.836 | 0.833 |
| frozen MiniLM | 0.791 | 0.787 |
| TF-IDF+SVM | 0.835 | 0.833 |
| BoW+chi²+BernoulliNB (Zhang's method) | 0.837 | 0.829 |
| **SciBERT-FT** | **0.876** | **0.880** |

Our frozen bge (0.836) and the classical lexical models (~0.835) **match the honest Zhang SOTA (0.825)**; SciBERT-FT **exceeds** it.

## Benchmark 2 — SAL/RCT (binary limitation)
~20% positive, paper-disjoint. SOTA Lan ~0.81.

In [ ]:
sal=pd.read_csv(EXT/"sal_rct/train.csv"); sal["y"]=sal["limitation_annotation"].astype(str).str.contains("True").map({True:"limitation",False:"neg"})
lim=sal[sal.y=="limitation"]; neg=sal[sal.y=="neg"].sample(min(len(lim)*4,(sal.y=="neg").sum()),random_state=0)
d=pd.concat([lim,neg]); d=pd.DataFrame({"text":d.sentences.astype(str),"y":d.y.values,"group":d.pmid.astype(str)}); d=d[d.text.str.len()>=20]
tr,te=split(d); print("SAL",len(d),"test",len(te))
compare(tr,te,["neg","limitation"],"limitation",_LIM,frozen=True)

**Reference (H100, full run):**
| model | macro_F1 | pos_F1(limitation) |
|---|---|---|
| cue-rules | 0.581 | 0.256 |
| frozen bge-small | 0.813 | 0.710 |
| frozen MiniLM | 0.729 | 0.584 |
| TF-IDF+SVM | 0.833 | 0.735 |
| BoW+chi²+BernoulliNB | 0.834 | 0.729 |
| **SciBERT-FT** | **0.900** | **0.843** |

SciBERT-FT (0.900) **meets/beats Lan's ~0.81**; classical and frozen bge are solid mid-0.8 macro-F1.

## Benchmark 3 — Sci-Sentence (3-class: none / limitation / future_work)
Collapsed from their 7 classes, paper-disjoint 80/20 on 700 sentences. SOTA Bolaños >0.96 (their 7-class split + 2,240 augmented).

In [ ]:
sci=pd.read_excel(EXT/"scisentence_repo/datasets/Based_Version.xlsx")
M={"LIMITATION":"limitation","EXTENSION":"future_work","RESEARCH GAP":"future_work","OVERALL":"none","DESCRIPTION":"none","RESULT":"none","OTHER":"none"}
d=pd.DataFrame({"text":sci.Sentence.astype(str),"y":sci.Category.str.upper().map(lambda c:M.get(c,"none")),"group":sci.get("Title",sci.Id).astype(str)})
tr,te=split(d); print("Sci-Sentence",len(d),"test",len(te))
compare(tr,te,["none","limitation","future_work"],None,None,frozen=True)

**Reference (H100, full run):**
| model | macro_F1 | lim_F1 | fw_F1 |
|---|---|---|---|
| frozen bge-small | 0.714 | 0.744 | 0.643 |
| frozen MiniLM | 0.622 | 0.667 | 0.517 |
| TF-IDF+SVM | 0.652 | 0.458 | 0.684 |
| BoW+chi²+BernoulliNB | 0.358 | 0.087 | 0.356 |
| **SciBERT-FT** | **0.856** | 0.706 | 0.916 |

SciBERT-FT best (0.856); below the paper's 0.96 because they use the full 7-class task, 2,240 augmented sentences, and stronger/larger models — **not directly comparable**.

## The key insight (reconciles the gold result)

| scenario | train vs test | best model type |
|---|---|---|
| **These benchmarks** | **same** dataset (in-domain) | **SciBERT fine-tune** (0.876 / 0.900 / 0.856) |
| **Our pipeline gold** | **different** domains (train SciSent+FWS+SAL → test arXiv) | **frozen head** (fine-tunes overfit) |

Fine-tuning wins **when you have in-domain training labels for the target** — exactly the benchmark setting,
where SciBERT-FT matches or beats the published SOTA. Frozen embeddings win for **cross-domain transfer** —
our real pipeline, where there is no arXiv gap-labelled training data. Both results are correct; they answer
different questions.

## Takeaways
- **On the public benchmarks, our pipeline is competitive/SOTA:** SciBERT-FT ≥ Zhang (FWS) and ≥ Lan (SAL); frozen bge ≈ honest Zhang SOTA with no training on neural hardware.
- **Zhang's 0.907 is leakage-inflated;** the fair target is 0.825, which we clear.
- **Classical BoW+chi²+BernoulliNB (Zhang's own method) reproduces ~0.84 on FWS/SAL** but collapses on 3-class Sci-Sentence (0.358) — it doesn't scale to multi-class.
- To actually run: set `RUN_SCIBERT=True` (GPU) and `RUN_FROZEN_FWS=True` for the full tables. Reference numbers above are from the H100 run (`v100_results/model_comparison.json`).